In [1]:
import os
import matplotlib
import pandas as pd
import xarray as xr
import rioxarray
from itertools import product
from dask.distributed import Client, LocalCluster
import platform
import numpy as np
import geopandas as gpd

Define paths

In [2]:
data_path = 'F:/Wetland_Climate_Impacts/Climate_Data/USGS_Thresholds_Weighted/Grid/pr/all_metrics'
meta_data_file = 'F:/Wetland_Climate_Impacts/Climate_Data/USGS_Water_Balance/NCCV2_LOCA2_model_extended_meta.csv'

Specify weight type

In [3]:
weight_type = 'NCA5_BMA_weight' # 'LOCA2_BMA_weight'

GHG emission scenarios

In [4]:
scenarios = np.array(['ssp245','ssp370','ssp585'])
n_s = len(scenarios)

Future year intervals

In [5]:
start_dates = np.array(['2041-01-01','2071-01-01'])
end_dates = np.array(['2070-12-31','2100-12-31'])
intervals = np.array(['2041_2070','2071_2100'])
n_y = len(intervals)

Define ensemble function

In [6]:
def create_ensemble(scenario, interval_index): 
    # Load the metadata file, filter only for our selected SSP, and create a unqiue ensemble ID (model.scenario.ripf).
    df = pd.read_csv(meta_data_file, encoding="ISO-8859-1")
    df = df[df['EXP'] == scenario]
    df['ensemble'] = df['MODEL']+"."+df['EXP']+"."+df['RIPF']
    
    # Drop most of the DataFrame and only keep the weights we want and the ensemble ID (also making it an index)
    df_weight = df.set_index('ensemble')[weight_type]
    
    # Since the ensemble ID is an index, we can easily convert this Pandas DataFrame to a xarray DataArray.
    weight_da = df_weight.to_xarray()
    
    # Use Python expansions to build our list of files
    hist_files = [
        f'{data_path}/historical/{ensemble_id.replace(scenario, 'historical')}.1950-2014.LOCA_16thdeg_pr_metrics.nc'
        for ensemble_id in list(weight_da.ensemble.values)
    ]
    
    ssp_files = [
        f'{data_path}/{scenario}/{ensemble_id}.2015-2100.LOCA_16thdeg_pr_metrics.nc'
        for ensemble_id in list(weight_da.ensemble.values)
    ]
    
    # Using the preprocessor, we can now load and combine all historical files.
    ds_hist = xr.open_mfdataset(hist_files, 
                                chunks={}, 
                                combine_attrs="drop_conflicts")
    
    # We do the same for the selected SSP files
    ds_ssp = xr.open_mfdataset(ssp_files, 
                               chunks={}, 
                               combine_attrs="drop_conflicts")
    
    # Rename the historical ensemble ID from model.historical.ripf to model.ssp.ripf so they can be concatenated. 
    # We need the ensemble ID labels to match between historical and SSP, so we force them to use the SSP IDs.
    ds_hist['ensemble'] = ds_hist.ensemble.str.replace('historical', scenario)
    
    # Combine historical + ssp for one single time series, for all models, all variables. 
    # Add weights to dataset as well. Notice the weights xr.DataArray aligns -by ID- across the xr.DataSet. 
    # This makes sure we don't jumble up the order of the model data vs the model weights.
    # 
    # At this point the xr.Dataset will be rather large (75 Gb per variable). 
    # Dask will load chunks of data on the fly as it needs them. 
    # However, if your computer is running out of memory or struggling, you can reduce the chunk size (read the xarray docs). 
    # The default with these data are ~50 Mb / chunk, which is pretty reasonable.
    ds = xr.concat([ds_hist, ds_ssp], dim="time")
    ds['weights'] = weight_da
    
    # ---- rechunk as needed here; it is currently [1, 360, 158, 236] ~ 50 Mb/chunk. 
    # ---- You could go higher or lower depending on the performance or amount of memory you have.
    
    # We can now create a weighted xr.Dataset for each year in the dataset. 
    # The weights var can be dropped because it is no longer needed.
    # weighted_ensemble_ds = ds.weighted(ds.weights).mean(dim="ensemble", keep_attrs=True).drop_vars("weights")
    
    # However, we need to be a little careful since the order of operation often matters. 
    # For example, if you want to create the weighted ensemble climatology, 
    # it is likely best to calculate the climatology first for each model, then apply the multimodel weights.
    ds_clim_historical = ds.sel(time=slice('1981-01-01','2010-12-31')).mean(dim="time")
    ds_clim_future = ds.sel(time=slice(start_dates[interval_index], end_dates[interval_index])).mean(dim="time")
    weighted_ds_clim_historical = ds_clim_historical.weighted(ds_clim_historical.weights).mean(dim="ensemble", keep_attrs=True).drop_vars("weights")
    weighted_ds_clim_future = ds_clim_future.weighted(ds_clim_future.weights).mean(dim="ensemble", keep_attrs=True).drop_vars("weights")
    weighted_ds_clim_change = weighted_ds_clim_future - weighted_ds_clim_historical
    
    # plot example
    #weighted_ds_clim_change.prmax1day.plot()
    
    # You are pretty much done at this point. You could save variables out to a new NetCDF or just do plotting. 
    # As I showed above, I would favor doing whatever averaging or summary you need to do first, 
    # and then apply the ensemble multimodel weights as the last step.
    # 
    # You are probably fine to spatially average grids -> HUCs using the weighted ensemble, 
    # but I would do temporal averaging before the model weighting just in case the order of operation matters.
    start_year = start_dates[interval_index].split('-')[0]
    end_year = end_dates[interval_index].split('-')[0]
    for v in list(weighted_ds_clim_change.keys())[1:len(weighted_ds_clim_change.keys())]:
        nc_filename = "F:/Wetland_Climate_Impacts/Climate_Data/USGS_Thresholds_Weighted/Grid/pr/bma_weighted_differences/{}/{}_{}/NCA5_BMA_Weighted_Ensemble_Difference_{}_{}_{}_{}.nc".format(scenario, start_year, end_year, v, scenario, start_year, end_year)
        weighted_ds_clim_change.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
        weighted_ds_clim_change.rio.write_crs("EPSG:4326", inplace=True) 
        weighted_ds_clim_change.rio.write_crs(4326,inplace=True).rio.set_spatial_dims(
            x_dim="lon",
            y_dim="lat",
            inplace=True
        ).rio.write_coordinate_system(inplace=True)
        weighted_ds_clim_change[v].to_netcdf(nc_filename)

Run each scenario and year interval combination

In [7]:
hostname = platform.uname()[1]
cluster = LocalCluster() 
client = Client(cluster)
print(f"Dask Dashboard is available at: {client.dashboard_link.replace("127.0.0.1", hostname)}")

Dask Dashboard is available at: http://SpudsCouch:8787/status


2026-02-18 18:11:09,168 - tornado.application - ERROR - Uncaught exception GET /status/ws (10.0.0.11)
HTTPServerRequest(protocol='http', host='spudscouch:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='10.0.0.11')
Traceback (most recent call last):
  File "C:\Users\Chels\anaconda3\envs\Climate_Ensembles\Lib\site-packages\tornado\websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "C:\Users\Chels\anaconda3\envs\Climate_Ensembles\Lib\site-packages\tornado\web.py", line 3388, in wrapper
    return method(self, *args, **kwargs)
  File "C:\Users\Chels\anaconda3\envs\Climate_Ensembles\Lib\site-packages\bokeh\server\views\ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-token-expiratio

In [8]:
for s in scenarios:
    for index in range(n_y):
        create_ensemble(s, index)

In [9]:
client.close()